In [ ]:
# ==========================================================
# PEDIATRIC DIABETES PREDICTION
# ==========================================================

!pip install imbalanced-learn -q

# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

from imblearn.over_sampling import RandomOverSampler

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import ExtraTreeClassifier

from sklearn.naive_bayes import GaussianNB

from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import *

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv('/content/Pediatrics.csv')

print("Original Shape")
print(df.shape)

# ==========================================================
# REMOVE DUPLICATES
# ==========================================================

df = df.drop_duplicates()

# ==========================================================
# HANDLE MISSING VALUES
# ==========================================================

df.fillna(df.mode().iloc[0], inplace=True)

# ==========================================================
# LABEL ENCODING
# ==========================================================

encoder = LabelEncoder()

for col in df.columns:

    if df[col].dtype == 'object':

        df[col] = encoder.fit_transform(df[col])

# ==========================================================
# CORRELATION HEATMAP
# ==========================================================

plt.figure(figsize=(15,10))

corr = df.corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title(
    "Correlation Heatmap — Pediatrics.csv",
    fontsize=14
)

plt.tight_layout()
plt.show()

# ==========================================================
# TARGET VARIABLE
# ==========================================================

TARGET = "Type of Diabetes"

# ==========================================================
# CORRELATION FEATURE SELECTION
# ==========================================================

corr_target = abs(corr[TARGET])

selected = corr_target[
    corr_target > 0.10
].index

selected = selected.drop(TARGET)

print("\nSelected Features")
print(selected)

# ==========================================================
# DEFINE X AND y
# ==========================================================

X = df[selected]

y = df[TARGET]

# ==========================================================
# BEST-K FEATURE SELECTION
# ==========================================================

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=min(14,len(selected))
)

X = selector.fit_transform(X,y)

feature_names = selected[
    selector.get_support()
]

print("\nSelected Top Features")
print(feature_names)

# ==========================================================
# FEATURE SCALING
# ==========================================================

scaler = StandardScaler()

X = scaler.fit_transform(X)

# ==========================================================
# HANDLE IMBALANCE
# ==========================================================

ros = RandomOverSampler(
    random_state=42
)

X,y = ros.fit_resample(
    X,
    y
)

print("\nClass Distribution")
print(pd.Series(y).value_counts())

# ==========================================================
# SPLIT DATA
# ==========================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nTrain:",len(X_train))
print("Validation:",len(X_val))
print("Test:",len(X_test))

# ==========================================================
# MODELS
# ==========================================================

models = {

    "RF": RandomForestClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    ),

    "RT": ExtraTreeClassifier(
        max_depth=15,
        random_state=42
    ),

    "DT": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),

    "NB": GaussianNB(),

    "LR": LogisticRegression(
        C=10,
        max_iter=3000,
        random_state=42
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(100,50),
        activation='relu',
        solver='adam',
        max_iter=3000,
        random_state=42
    )
}

# ==========================================================
# EVALUATION
# ==========================================================

results = []

for name, model in models.items():

    print(f"\nTraining {name}")

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        pred
    )

    precision = precision_score(
        y_test,
        pred,
        average='weighted'
    )

    recall = recall_score(
        y_test,
        pred,
        average='weighted'
    )

    f1 = f1_score(
        y_test,
        pred,
        average='weighted'
    )

    kappa = cohen_kappa_score(
        y_test,
        pred
    )

    mcc = matthews_corrcoef(
        y_test,
        pred
    )

    try:

        prob = model.predict_proba(
            X_test
        )

        roc = roc_auc_score(
            pd.get_dummies(y_test),
            prob,
            multi_class='ovr'
        )

    except:

        roc = np.nan

    results.append([

        name,
        round(accuracy,3),
        round(precision,3),
        round(recall,3),
        round(f1,3),
        round(roc,3),
        round(kappa,3),
        round(mcc,3)

    ])

# ==========================================================
# RESULTS TABLE
# ==========================================================

results_df = pd.DataFrame(

    results,

    columns=[

        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC",
        "Kappa",
        "MCC"

    ]

)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\nMODEL PERFORMANCE")
display(results_df)

# ==========================================================
# SAVE TABLE
# ==========================================================

results_df.to_csv(
    "Model_Results.csv",
    index=False
)

# ==========================================================
# RANDOM FOREST FEATURE IMPORTANCE
# ==========================================================

rf = models["RF"]

importance = rf.feature_importances_

importance_df = pd.DataFrame({

    "Feature": feature_names,
    "Importance": importance

})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

top14 = importance_df.head(14)

print("\nFeature Importance")
print(top14)

# ==========================================================
# FEATURE IMPORTANCE GRAPH
# ==========================================================

plt.figure(figsize=(8,6))

plt.barh(
    top14["Feature"],
    top14["Importance"]
)

plt.gca().invert_yaxis()

plt.title(
    "Top 14 Attribute Importance — Pediatrics.csv"
)

plt.xlabel("Importance")

plt.tight_layout()

plt.show()

# ==========================================================
# SAVE FEATURE IMPORTANCE
# ==========================================================

top14.to_csv(
    "Feature_Importance.csv",
    index=False
)

# ==========================================================
# RANDOM FOREST CONFUSION MATRIX
# ==========================================================

rf_pred = rf.predict(X_test)

cm = confusion_matrix(
    y_test,
    rf_pred
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title(
    "Random Forest Confusion Matrix"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

print("\nAnalysis Completed Successfully")